In [ ]:
# ============================================================
# CELL 1 — SETUP, CONSTANTS, MODEL LOAD
# NVIDIA Nemotron Reasoning Challenge — Champion Strategy
# ============================================================

# ----- 1.1  Install dependencies ----------------------------
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("unsloth[colab-new]")          # fastest LoRA, ~40% less VRAM than HF vanilla
pip("trl>=0.12.0")                 # GRPOTrainer with asymmetric clip support
pip("vllm==0.6.6")                 # MUST match competition inference version
pip("wandb")                       # experiment tracking — never debug blind
pip("datasets", "accelerate", "peft")

# ----- 1.2  Imports -----------------------------------------
import os
import re
import json
import math
import random
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import wandb
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from trl import SFTTrainer, GRPOTrainer, GRPOConfig

# ----- 1.3  Reproducibility ---------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ----- 1.4  Competition constants ---------------------------
@dataclass
class CompCfg:
    # Model
    base_model: str        = "nvidia/Nemotron-3-Nano-30B"  # or local path
    max_seq_len: int       = 8192     # competition max_model_len
    max_new_tokens: int    = 7680     # competition max_tokens

    # Nemotron-3-Nano think token IDs (Mamba-Transformer MoE)
    think_open_id: int     = 12       # <think>
    think_close_id: int    = 13       # </think>

    # LoRA — competition hard limits
    lora_rank: int         = 32       # hard limit from competition rules
    lora_alpha: int        = 64       # 2× rank for strong updates
    lora_dropout: float    = 0.05
    lora_target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP  ← critical for reasoning
    ])

    # SFT hyperparameters
    sft_lr: float          = 1e-4     # 10× full-FT LR — LoRA rule
    sft_epochs: int        = 3
    sft_batch_size: int    = 4
    sft_grad_accum: int    = 8        # effective batch = 32
    sft_warmup_steps: int  = 50

    # DAPO / RL hyperparameters
    rl_lr: float           = 1e-6     # 10× smaller than SFT — prevent catastrophic forgetting
    rl_group_size: int     = 16       # G rollouts per prompt (DeepSeek-R1 validated)
    rl_clip_low: float     = 0.2      # standard lower clip
    rl_clip_high: float    = 0.28     # DAPO Clip-Higher — prevents entropy collapse
    rl_kl_coeff: float     = 0.001    # very small — allow long CoT chains to grow
    rl_rollout_temp: float = 1.0      # MUST be 1.0 for diverse exploration during training
    rl_eval_temp: float    = 0.0      # greedy for evaluation

    # Paths
    data_dir: str          = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge"
    output_dir: str        = "/kaggle/working/nemotron-adapter"
    hf_repo: str           = ""       # set to push checkpoints to HuggingFace Hub

cfg = CompCfg()

# ----- 1.5  W&B init ----------------------------------------
wandb.init(
    project="nemotron-reasoning-challenge",
    config={
        "lora_rank": cfg.lora_rank,
        "lora_alpha": cfg.lora_alpha,
        "sft_lr": cfg.sft_lr,
        "rl_lr": cfg.rl_lr,
        "rl_group_size": cfg.rl_group_size,
        "rl_clip_low": cfg.rl_clip_low,
        "rl_clip_high": cfg.rl_clip_high,
        "rl_kl_coeff": cfg.rl_kl_coeff,
        "seed": SEED,
    },
    mode="online",   # set to "disabled" for offline runs
)

# ----- 1.6  Load model + LoRA adapter -----------------------
print(f"Loading {cfg.base_model} with Unsloth...")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.base_model,
    max_seq_length=cfg.max_seq_len,
    dtype=None,          # auto-detect: fp16 on RTX, bfloat16 on H100
    load_in_4bit=False,  # full fp16 LoRA — use True only if VRAM < 48GB
)

model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_rank,
    target_modules=cfg.lora_target_modules,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",  # reduces activation memory ~5×
    random_state=SEED,
    use_rslora=True,    # rank-stabilised LoRA — more stable at rank=32
)

# ----- 1.7  Verify think token IDs --------------------------
think_open  = tokenizer.convert_tokens_to_ids("<think>")
think_close = tokenizer.convert_tokens_to_ids("</think>")
print(f"<think> token ID  : {think_open}  (expected: {cfg.think_open_id})")
print(f"</think> token ID : {think_close} (expected: {cfg.think_close_id})")
assert think_open  == cfg.think_open_id,  f"<think> ID mismatch: got {think_open}"
assert think_close == cfg.think_close_id, f"</think> ID mismatch: got {think_close}"

# ----- 1.8  Helper: answer extraction & matching ------------
def extract_boxed_answer(text: str) -> Optional[str]:
    """Extract content from \\boxed{...}, handles nested braces."""
    idx = text.rfind(r"\boxed{")
    if idx == -1:
        return None
    depth, start = 0, idx + len(r"\boxed{")
    for i, ch in enumerate(text[idx:], start=idx):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i].strip()
    return None

def answers_match(pred: str, truth: str) -> bool:
    """Exact string match OR relative numerical tolerance of 1e-6."""
    if pred.strip() == truth.strip():
        return True
    try:
        p, t = float(pred.strip()), float(truth.strip())
        denom = max(abs(t), 1e-9)
        return abs(p - t) / denom < 1e-6
    except (ValueError, TypeError):
        return False

def compute_reward(response: str, ground_truth: str, step: int = 9999) -> float:
    """RLVR reward: verifiable accuracy + format. No length penalty (proven harmful)."""
    has_boxed     = r"\boxed{" in response
    fmt_reward    = 0.1 if has_boxed else 0.0
    answer        = extract_boxed_answer(response)

    if answer is None:
        return fmt_reward * 0.5

    acc_reward = 1.0 if answers_match(answer, ground_truth) else 0.0

    # Fade format reward after step 500 — model should rely on accuracy signal alone
    fmt_weight = max(0.0, 1.0 - step / 500.0)
    return acc_reward + fmt_reward * fmt_weight

# ----- 1.9  Prompt template ---------------------------------
SYSTEM_PROMPT = (
    "You are a precise logical reasoning assistant. "
    "Think step by step inside <think> tags. "
    "Always end with your final answer inside \\boxed{}."
)

def build_prompt(puzzle: str) -> str:
    """Format a puzzle into the competition prompt structure."""
    return tokenizer.apply_chat_template(
        [
            {"role": "system",  "content": SYSTEM_PROMPT},
            {"role": "user",    "content": puzzle},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

# ----- 1.10 Quick sanity check ------------------------------
dummy_output = (
    "<think>\nLet me apply the XOR rule: 5 XOR 3 = 6\n</think>\n\\boxed{6}"
)
assert extract_boxed_answer(dummy_output) == "6",          "extract_boxed broken"
assert answers_match("6", "6"),                            "exact match broken"
assert answers_match("6.0000001", "6") == False,           "tolerance too loose"
assert compute_reward(dummy_output, "6", step=0)  > 1.0,   "reward should be >1.0 at step 0"
assert compute_reward(dummy_output, "99", step=0) < 0.5,   "wrong answer should score low"

print("\n✓ All sanity checks passed.")
print(f"✓ Model loaded: {cfg.base_model}")
print(f"✓ LoRA rank={cfg.lora_rank}, alpha={cfg.lora_alpha}")
print(f"✓ Think tokens verified: <think>={think_open}, </think>={think_close}")
model.print_trainable_parameters()
